### 93. 复原 IP 地址
有效 IP 地址 正好由四个整数（每个整数位于 0 到 255 之间组成，且不能含有前导 0），整数之间用 '.' 分隔。

例如："0.1.2.201" 和 "192.168.1.1" 是 有效 IP 地址，但是 "0.011.255.245"、"192.168.1.312" 和 "192.168@1.1" 是 无效 IP 地址。
给定一个只包含数字的字符串 s ，用以表示一个 IP 地址，返回所有可能的有效 IP 地址，这些地址可以通过在 s 中插入 '.' 来形成。你 不能 重新排序或删除 s 中的任何数字。你可以按 任何 顺序返回答案。

示例 1：

输入：s = "25525511135"
输出：["255.255.11.135","255.255.111.35"]

示例 2：

输入：s = "0000"
输出：["0.0.0.0"]

示例 3：
输入：s = "101023"
输出：["1.0.10.23","1.0.102.3","10.1.0.23","10.10.2.3","101.0.2.3"]

#### 1. 回溯（选或不选）
本题要求分成恰好 4 段。

在 131 题目基础上，我们需要额外知道：
- 当前在第几段。
- 当前分割的子串，对应的数值是多少，便于我们判断子串是否合法。

定义 dfs(i,j,ipVal)，表示：
- 剩余字符从 s[i] 到 s[n−1]。
- 现在在第 j 段（j 从 0 开始）。
- 第 j 段的数值目前为 ipVal。

递归过程中，首先把 s[i] 加到当前这一段的末尾，即更新 ipVal 为 ipVal⋅10+int(s[i])。例如在 12 的末尾添加 3，数值更新为 12⋅10+3=123。

分类讨论：
- 不分割，前提是不能有前导零，即此时 ipVal>0。往下递归到 dfs(i+1,j,ipVal)。注意，如果有前导零的话，会在前导零那个字符处发现 ipVal=0，不会往下递归。
- 分割，s[i] 作为当前这段子串的右端点。把 j 加一，ipVal 重置为 0。往下递归到 dfs(i+1,j+1,0)。

递归结束条件：
- i=n 时，s 分割完毕，如果此时 j=4，把分割结果加入答案。
- 否则，如果 j=4，由于此时已经分出 4 段，不能再分割，所以不再递归。
递归入口：dfs(0,0,0)。

In [ ]:
# path 的长度是固定的 4，可以直接覆盖 path[j]，无需恢复现场。
# 如果是 path 初始化为空列表的那种写法，就需要恢复现场。

class Solution:
    def restoreIpAddresses(self, s: str) -> list[str]:
        n = len(s)
        ans = []
        path = [0] * 4 # path[i] 表示第 i 段（i 从 0 开始）的结束位置 + 1（右开区间，方便切片）

        # 分割 s[i] 到 s[n-1]， 现在在第 j段(j 从 0 开始),数值为ip_val
        def dfs(i, j, ip_val) -> None:
            if i == n:
                if j == 4: # 分割完毕且有4段，递归终止条件
                    a, b, c, _ = path
                    ans.append(f"{s[:a]}.{s[a:b]}.{s[b:c]}.{s[c:]}") # 拼接 path
                return 

            if j == 4: # 已经有4段了，但还没分割完，不能有甚于字符，直接返回
                return

            # 手动把字符串转成整数，这样字符串转整数是 严格 O(1)的
            ip_val = ip_val * 10 + int(s[i]) # 计算当前段的数值
            if ip_val > 255: # 数值超过 255，不合法
                return

            # 不分割，不以 s[i]结尾，继续
            if ip_val > 0: #无前导零
                dfs(i+1, j, ip_val)

            # 分割，以 s[i] 为这一段的结尾
            path[j] = i + 1  # 记录这一段的结束位置 + 1
            dfs(i+1, j+1, 0) 

        dfs(0, 0, 0)
        return ans
        